# Extração das Bases

In [ ]:
import os
import boto3
from dotenv import load_dotenv

## Configurações do Bucket

In [ ]:
NOME_DO_BUCKET = 'treinamento2-2026.0'
REGIAO = 'us-east-2'
PASTA_DESTINO = 'bases_baixadas'

## Acesso às credenciais

In [ ]:
def baixar_todas_as_bases():
    load_dotenv()
    aws_id = os.getenv('AWS_ACCESS_KEY_ID')
    aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')

    s3_client = boto3.client(
        's3',
        region_name=REGIAO,
        aws_access_key_id=aws_id,
        aws_secret_access_key=aws_secret
    )

    if not os.path.exists(PASTA_DESTINO):
        os.makedirs(PASTA_DESTINO)

    try:
        response = s3_client.list_objects_v2(Bucket=NOME_DO_BUCKET)
        
        for item in response['Contents']:
            nome_arquivo = item['Key']
            
            if nome_arquivo.endswith('/'):
                continue
                
            caminho_local = os.path.join(PASTA_DESTINO, nome_arquivo)
            
            subpasta_local = os.path.dirname(caminho_local)
            if not os.path.exists(subpasta_local):
                os.makedirs(subpasta_local)

            s3_client.download_file(NOME_DO_BUCKET, nome_arquivo, caminho_local)
        
    except Exception as e:
        return

In [ ]:
if __name__ == "__main__":
    baixar_todas_as_bases()

# Tratamento de Dados

## Tratamento das Bases Extraídas

In [ ]:
import pandas as pd
import numpy as np

### Tratando base_financeiro
1. Conferir se todas as células na colunas DAYS possuíam números negativos.


In [ ]:
caminho_completo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_baixadas/base_financeiro.csv'
df = pd.read_csv(caminho_completo) 

colunas_days = [col for col in df.columns if 'DAYS' in col]

df[colunas_days] = df[colunas_days].mask(df[colunas_days] > 0, np.nan)

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_financeiro_limpa.csv'
df.to_csv(caminho_salvar, index=False)

### Tratando base_infos_pessoais
1. Conferir se todas as células na colunas DAYS possuíam números negativos;
2. Conferir se todo indivíduo possuía CNT_FAM_MEMBERS = 1 (se solteiro)/2 (se casado) + CNT_CHILDRE
3. Conferir se todo inidivíduo era maior de idade;
4. Conferir se o nível de escolaridade do indivíduo era adequado à sua idade;

In [ ]:
caminho_novo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_baixadas/base_infos_pessoais.csv'
df_novo = pd.read_csv(caminho_novo)

df_novo.loc[df_novo['DAYS_BIRTH'] >= 0, 'DAYS_BIRTH'] = np.nan

mask_single = df_novo['NAME_FAMILY_STATUS'] == 'Single / Not Married'
df_novo.loc[mask_single, 'CNT_CHILDREN'] = 0
df_novo.loc[mask_single, 'CNT_FAM_MEMBERS'] = 1 + df_novo.loc[mask_single, 'CNT_CHILDREN']

mask_married = df_novo['NAME_FAMILY_STATUS'].isin(['Married', 'Civil Marriage'])
df_novo.loc[mask_married, 'CNT_FAM_MEMBERS'] = 2 + df_novo.loc[mask_married, 'CNT_CHILDREN']

df_novo.loc[df_novo['DAYS_BIRTH'] >= -6574, 'DAYS_BIRTH'] = np.nan

mask_higher = (df_novo['NAME_EDUCATION_TYPE'] == 'Higher education') & (df_novo['DAYS_BIRTH'] > -7665)
df_novo.loc[mask_higher, 'DAYS_BIRTH'] = np.nan

colunas_verificacao = ['DAYS_BIRTH', 'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_EDUCATION_TYPE']

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_infos_pessoais_limpa.csv'

df_novo.to_csv(caminho_salvar, index=False)

### Tratando base_regional
1. Conferir se nas colunas cabíveis os valores variavam apenas entre 0 e 1;
2. Conferir se nas colunas de rating os valores variavam apenas de 1 a 3;

In [ ]:
caminho_regional = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_baixadas/base_regional.csv'
df_regional = pd.read_csv(caminho_regional)

colunas_0_1 = [
    'REGION_POPULATION_RELATIVE', 
    'APARTMENTS_AVG', 
    'YEARS_BEGINEXPLUATATION_AVG', 
    'ELEVATORS_AVG'
]

for col in colunas_0_1:
    df_regional[col] = df_regional[col].mask((df_regional[col] < 0) | (df_regional[col] > 1), np.nan)

col_rating = 'REGION_RATING_CLIENT_W_CITY'

df_regional[col_rating] = df_regional[col_rating].mask((df_regional[col_rating] < 1) | (df_regional[col_rating] > 3), np.nan)

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_regional_limpa.csv'
df_regional.to_csv(caminho_salvar, index=False)

### Tratando base_scores
1. Conferindo se, nas colunas cabíveis, os valores estão entre 0 e 1;
2. Conferindo se nas colunas de dias os números são negativos;

In [ ]:
path_scores = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_baixadas/base_scores.csv'
df_scores = pd.read_csv(path_scores)

cols_ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for col in cols_ext:
    df_scores[col] = df_scores[col].mask((df_scores[col] < 0) | (df_scores[col] > 1), np.nan)

df_scores['DAYS_LAST_PHONE_CHANGE'] = df_scores['DAYS_LAST_PHONE_CHANGE'].mask(df_scores['DAYS_LAST_PHONE_CHANGE'] >= 0, np.nan)

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_scores_limpa.csv'
df_scores.to_csv(caminho_salvar, index=False)

### Junção da Bases

In [ ]:
paths = [
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_baixadas/base_bens.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_financeiro_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_infos_pessoais_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_regional_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/base_scores_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_baixadas/base_target.csv'
]

df_final = pd.read_csv(paths[0])

for path in paths[1:]:
    df_proximo = pd.read_csv(path)
    
    df_final = pd.merge(df_final, df_proximo, on='SK_ID_CURR', how='left')

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/bases_unificadas.csv'
df_final.to_csv(caminho_salvar, index=False)

## Ordinal e One-hot Encolding 

In [ ]:
caminho_arquivo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/bases_unificadas.csv'
df = pd.read_csv(caminho_arquivo)

df.drop(columns=['SK_ID_CURR'], inplace=True)

mapping_yes_no = {'Yes': 0, 'No': 1, 'Y': 0, 'N': 1}

df['FLAG_OWN_CAR'] = df['FLAG_OWN_CAR'].map(mapping_yes_no)
df['FLAG_OWN_REALTY'] = df['FLAG_OWN_REALTY'].map(mapping_yes_no)

df['APARTMENTS_AVG'] = df['APARTMENTS_AVG'].fillna(0)
df['YEARS_BEGINEXPLUATATION_AVG'] = df['YEARS_BEGINEXPLUATATION_AVG'].fillna(0)
df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'].fillna(0)

df['CODE_GENDER'] = df['CODE_GENDER'].map({'F': 0, 'M': 1, 'Female': 0, 'Male': 1})

cols_positivas = ['DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_BIRTH', 'DAYS_LAST_PHONE_CHANGE']

for col in cols_positivas:
    df[col] = df[col].abs()

cols_mediana = ['OWN_CAR_AGE', 'DAYS_ID_PUBLISH', 'DAYS_EMPLOYED', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'DAYS_LAST_PHONE_CHANGE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for col in cols_mediana:
    df[col] = df[col].fillna(df[col].median())

cols_one_hot = ['NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_FAMILY_STATUS']

df = pd.get_dummies(df, columns=[c for c in cols_one_hot if c in df.columns])

edu_mapping = {
    'Lower secondary': 1,
    'Secondary / secondary special': 2,
    'Incomplete higher': 3,
    'Higher education': 4,
    'Academic degree': 5  
}
if 'NAME_EDUCATION_TYPE' in df.columns:
    df['NAME_EDUCATION_TYPE'] = df['NAME_EDUCATION_TYPE'].map(edu_mapping)

df.to_csv('/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_analise_exploratoria/bases_unificadas_encoding.csv', index=False)

# Análise Exploratória

In [ ]:
import phik
from phik import resources, report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

caminho_arquivo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_analise_exploratoria/bases_unificadas_encoding.csv'
df = pd.read_csv(caminho_arquivo)

## Análises por Correlação

### Correlação de Pearson

In [ ]:
coluna_alvo = 'TARGET_CREDIT_LIMIT'

print(f"\n--- ANÁLISE REAL DO ALVO: {coluna_alvo} ---")

correlacoes_limite = df.corr()[coluna_alvo].sort_values(ascending=False)

print(correlacoes_limite[1:11]) 

print(correlacoes_limite.tail(10))

### Correlação Phik

In [ ]:
df_amostra = df.sample(min(10000, len(df))) 

matriz_phik = df_amostra.phik_matrix()

phik_target = matriz_phik['TARGET_CREDIT_LIMIT'].sort_values(ascending=False)

### Corelação de Spearman (ignorando Dummies)

In [ ]:
prefixos_excluir = [
    'NAME_HOUSING_TYPE_', 
    'NAME_INCOME_TYPE_', 
    'OCCUPATION_TYPE_', 
    'ORGANIZATION_TYPE_', 
    'NAME_FAMILY_STATUS_'
]

cols_analise = [
    col for col in df.columns 
    if (df[col].dtype in ['int64', 'float64', 'int32', 'bool']) 
    and not any(col.startswith(p) for p in prefixos_excluir)
]

if 'TARGET_CREDIT_LIMIT' not in cols_analise:
    if 'TARGET_CREDIT_LIMIT' in df.columns:
        cols_analise.append('TARGET_CREDIT_LIMIT')
    else:
        cols_analise = []

if cols_analise:
    corr_spearman = df[cols_analise].corr(method='spearman')

    ranking = corr_spearman['TARGET_CREDIT_LIMIT'].sort_values(ascending=False).dropna()
    
    ranking_plot = ranking.drop('TARGET_CREDIT_LIMIT', errors='ignore')

    top_print = pd.concat([ranking_plot.head(10), ranking_plot.tail(5)])
    print(top_print)

    plt.figure(figsize=(12, 10))
    cores = ['red' if x < 0 else 'skyblue' for x in top_print]
    
    top_print.plot(kind='barh', color=cores)
    plt.title('Correlação de Spearman (Excluindo Dummies Categóricas)', fontsize=15)
    plt.xlabel('Coeficiente de Correlação')
    plt.ylabel('Variáveis')
    plt.axvline(0, color='black', linewidth=0.8) 
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.invert_yaxis() 
    plt.tight_layout()
    plt.show()

## Análise por Quadrantes (dummies)

### Função Estruturada

In [ ]:
def plot_quadrants_from_dummies(df, prefix, target="TARGET_CREDIT_LIMIT"):
    # pega dummies
    cols = [c for c in df.columns if c.startswith(prefix)]
    
    # cria categoria única
    data = df.copy()
    data["category"] = data[cols].idxmax(axis=1)
    data["category"] = data["category"].str.replace(prefix, "")
    
    # agrega
    agg = data.groupby("category").agg(
        mean_target=(target, "mean"),
        freq=("category", "count")
    )
    
    # cortes
    freq_cut = agg["freq"].median()
    target_cut = agg["mean_target"].median()
    
    # quadrante
    def get_q(row):
        if row["mean_target"] >= target_cut and row["freq"] >= freq_cut:
            return "Q1"
        elif row["mean_target"] >= target_cut:
            return "Q2"
        elif row["freq"] >= freq_cut:
            return "Q3"
        else:
            return "Q4"
    
    agg["quadrant"] = agg.apply(get_q, axis=1)
    
    # plot
    plt.figure(figsize=(10,8))
    
    sns.scatterplot(
        data=agg,
        x="freq",
        y="mean_target",
        hue="quadrant",
        size="freq",
        sizes=(50,400),
        alpha=0.7
    )
    
    plt.axvline(freq_cut, linestyle="--", color="black")
    plt.axhline(target_cut, linestyle="--", color="black")
    
    for i in agg.index:
        plt.text(agg.loc[i,"freq"], agg.loc[i,"mean_target"], i, fontsize=7)
    
    plt.xlabel("Frequência")
    plt.ylabel("Média TARGET")
    plt.title(f"Quadrantes - {prefix}")
    plt.tight_layout()
    plt.show()
    
    return agg.sort_values("quadrant")

### Estado Civil

In [ ]:
df = df.drop(columns=[col for col in df.columns if "Unknown" in col or "UNKNOWN" in col])
plot_quadrants_from_dummies(df, "NAME_FAMILY_STATUS")

### Tipo de Carreira

In [ ]:
plot_quadrants_from_dummies(df, "NAME_INCOME_TYPE")

### Tipo de Ocupação

In [ ]:
plot_quadrants_from_dummies(df, "OCCUPATION_TYPE")

### Tipo de Moradia

In [ ]:
plot_quadrants_from_dummies(df, "NAME_HOUSING_TYPE_")

### Tipo de Organização

In [ ]:
plot_quadrants_from_dummies(df, "ORGANIZATION_TYPE_")

## Tratamento Final da Base (33 Colunas para Treinamento do Modelo)

In [ ]:
fam_q1 = ['NAME_FAMILY_STATUS_Married', 'NAME_FAMILY_STATUS_Single / not married']
fam_q2 = ['NAME_FAMILY_STATUS_Separated']
fam_q3 = ['NAME_FAMILY_STATUS_Civil marriage']
fam_q4 = ['NAME_FAMILY_STATUS_Widow']

inc_q1 = ['NAME_INCOME_TYPE_Commercial associate', 'NAME_INCOME_TYPE_State servant']
inc_q2 = ['NAME_INCOME_TYPE_Businessman', 'NAME_INCOME_TYPE_Student']
inc_q3 = ['NAME_INCOME_TYPE_Pensioner', 'NAME_INCOME_TYPE_Working']
inc_q4 = ['NAME_INCOME_TYPE_Maternity leave', 'NAME_INCOME_TYPE_Unemployed']

occ_q1 = ['OCCUPATION_TYPE_Laborers', 'OCCUPATION_TYPE_Core staff', 'OCCUPATION_TYPE_Drivers', 'OCCUPATION_TYPE_High skill tech staff', 'OCCUPATION_TYPE_Managers']
occ_q2 = ['OCCUPATION_TYPE_Realty agents', 'OCCUPATION_TYPE_HR staff', 'OCCUPATION_TYPE_IT staff', 'OCCUPATION_TYPE_Private service staff']
occ_q3 = ['OCCUPATION_TYPE_Sales staff', 'OCCUPATION_TYPE_Medicine staff', 'OCCUPATION_TYPE_Accountants', 'OCCUPATION_TYPE_Security staff']
occ_q4 = ['OCCUPATION_TYPE_Cooking staff', 'OCCUPATION_TYPE_Cleaning staff', 'OCCUPATION_TYPE_Secretaries', 'OCCUPATION_TYPE_Low-skill Laborers', 'OCCUPATION_TYPE_Waiters/barmen staff']

hou_q1 = ['NAME_HOUSING_TYPE_House / apartment']
hou_q2 = ['NAME_HOUSING_TYPE_Co-op apartment', 'NAME_HOUSING_TYPE_Office apartment']
hou_q3 = ['NAME_HOUSING_TYPE_Municipal apartment', 'NAME_HOUSING_TYPE_With parents']
hou_q4 = ['NAME_HOUSING_TYPE_Rented apartment']

org_q1 = ['ORGANIZATION_TYPE_Police', 'ORGANIZATION_TYPE_Trade: type 2', 'ORGANIZATION_TYPE_Security Ministries', 'ORGANIZATION_TYPE_Industry: type 9', 'ORGANIZATION_TYPE_Transport: type 4', 'ORGANIZATION_TYPE_Other', 'ORGANIZATION_TYPE_Military', 'ORGANIZATION_TYPE_Services', 'ORGANIZATION_TYPE_Business Entity Type 3', 'ORGANIZATION_TYPE_Business Entity Type 2', 'ORGANIZATION_TYPE_Business Entity Type 1', 'ORGANIZATION_TYPE_Bank', 'ORGANIZATION_TYPE_Transport: type 2', 'ORGANIZATION_TYPE_Construction']
org_q2 = ['ORGANIZATION_TYPE_Realtor', 'ORGANIZATION_TYPE_Insurance', 'ORGANIZATION_TYPE_Industry: type 5', 'ORGANIZATION_TYPE_Industry: type 4', 'ORGANIZATION_TYPE_Legal Services', 'ORGANIZATION_TYPE_University', 'ORGANIZATION_TYPE_Telecom', 'ORGANIZATION_TYPE_Industry: type 10', 'ORGANIZATION_TYPE_Trade: type 5', 'ORGANIZATION_TYPE_Emergency', 'ORGANIZATION_TYPE_Electricity', 'ORGANIZATION_TYPE_Culture', 'ORGANIZATION_TYPE_Transport: type 1', 'ORGANIZATION_TYPE_Industry: type 2', 'ORGANIZATION_TYPE_Industry: type 12']
org_q3 = ['ORGANIZATION_TYPE_Self-employed', 'ORGANIZATION_TYPE_Trade: type 3', 'ORGANIZATION_TYPE_Security', 'ORGANIZATION_TYPE_School', 'ORGANIZATION_TYPE_Trade: type 7', 'ORGANIZATION_TYPE_Restaurant', 'ORGANIZATION_TYPE_Advertising', 'ORGANIZATION_TYPE_Kindergarten', 'ORGANIZATION_TYPE_Medicine', 'ORGANIZATION_TYPE_Agriculture', 'ORGANIZATION_TYPE_Government', 'ORGANIZATION_TYPE_Industry: type 3', 'ORGANIZATION_TYPE_Industry: type 11', 'ORGANIZATION_TYPE_Postal', 'ORGANIZATION_TYPE_Housing']
org_q4 = ['ORGANIZATION_TYPE_Transport: type 3', 'ORGANIZATION_TYPE_Cleaning', 'ORGANIZATION_TYPE_Trade: type 6', 'ORGANIZATION_TYPE_Hotel', 'ORGANIZATION_TYPE_Mobile', 'ORGANIZATION_TYPE_Trade: type 1', 'ORGANIZATION_TYPE_Industry: type 13', 'ORGANIZATION_TYPE_Industry: type 6', 'ORGANIZATION_TYPE_Industry: type 7', 'ORGANIZATION_TYPE_Industry: type 8', 'ORGANIZATION_TYPE_Religion', 'ORGANIZATION_TYPE_Trade: type 4', 'ORGANIZATION_TYPE_Industry: type 1']

def agrupar_quadrantes(df_original, palavras_chave):
    colunas_alvo = [
        col for col in df_original.columns 
        if any(p.lower() in col.lower() for p in palavras_chave)
    ]
    
    if colunas_alvo:
        resultado_bool = df_original[colunas_alvo].any(axis=1)
        
        return resultado_bool.astype(int)
    else:
        return pd.Series(0, index=df_original.index)

df['FAMILY_Q1'] = agrupar_quadrantes(df, ['Married', 'Single'])

df['FAMILY_Q1'] = agrupar_quadrantes(df, fam_q1)
df['FAMILY_Q2'] = agrupar_quadrantes(df, fam_q2)
df['FAMILY_Q3'] = agrupar_quadrantes(df, fam_q3)
df['FAMILY_Q4'] = agrupar_quadrantes(df, fam_q4)

df['INCOME_Q1'] = agrupar_quadrantes(df, inc_q1)
df['INCOME_Q2'] = agrupar_quadrantes(df, inc_q2)
df['INCOME_Q3'] = agrupar_quadrantes(df, inc_q3)
df['INCOME_Q4'] = agrupar_quadrantes(df, inc_q4)

df['OCCUPATION_Q1'] = agrupar_quadrantes(df, occ_q1)
df['OCCUPATION_Q2'] = agrupar_quadrantes(df, occ_q2)
df['OCCUPATION_Q3'] = agrupar_quadrantes(df, occ_q3)
df['OCCUPATION_Q4'] = agrupar_quadrantes(df, occ_q4)

df['HOUSING_Q1'] = agrupar_quadrantes(df, hou_q1)
df['HOUSING_Q2'] = agrupar_quadrantes(df, hou_q2)
df['HOUSING_Q3'] = agrupar_quadrantes(df, hou_q3)
df['HOUSING_Q4'] = agrupar_quadrantes(df, hou_q4)

df['ORG_Q1'] = agrupar_quadrantes(df, org_q1)
df['ORG_Q2'] = agrupar_quadrantes(df, org_q2)
df['ORG_Q3'] = agrupar_quadrantes(df, org_q3)
df['ORG_Q4'] = agrupar_quadrantes(df, org_q4)

df['IS_HIGH_EDUCATION'] = (df['NAME_EDUCATION_TYPE'] >= 3).astype(int)

colunas_finais = [
    'FLAG_OWN_CAR', 'REGION_RATING_CLIENT', 'AMT_INCOME_TOTAL', 
    'CNT_CHILDREN', 'IS_HIGH_EDUCATION', 'REGION_POPULATION_RELATIVE', 
    'REG_REGION_NOT_WORK_REGION', 'APARTMENTS_AVG', 'EXT_SOURCE_2', 
    'OBS_30_CNT_SOCIAL_CIRCLE', 'TARGET_CREDIT_LIMIT', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    
    'FAMILY_Q1', 'FAMILY_Q4',
    'INCOME_Q1', 'INCOME_Q2', 'INCOME_Q3',
    'OCCUPATION_Q1', 'OCCUPATION_Q2', 'OCCUPATION_Q3', 'OCCUPATION_Q4',
    'HOUSING_Q1', 'HOUSING_Q2', 'HOUSING_Q3', 'HOUSING_Q4',
    'ORG_Q1', 'ORG_Q2'
]

df = df[colunas_finais]

def filtrar_outliers_por_grupo(df_grupo):
    if len(df_grupo) < 5:
        return df_grupo
        
    Q1 = df_grupo['TARGET_CREDIT_LIMIT'].quantile(0.25)
    Q3 = df_grupo['TARGET_CREDIT_LIMIT'].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_superior = Q3 + 1.5 * IQR
    limite_inferior = Q1 - 1.5 * IQR
    
    return df_grupo[(df_grupo['TARGET_CREDIT_LIMIT'] >= limite_inferior) & 
                    (df_grupo['TARGET_CREDIT_LIMIT'] <= limite_superior)]

colunas_categoricas = [
    'CNT_CHILDREN', 
    'REGION_RATING_CLIENT', 
    'AMT_REQ_CREDIT_BUREAU_YEAR', 
    'OBS_30_CNT_SOCIAL_CIRCLE'
]

for coluna in colunas_categoricas:
    df = df.groupby(coluna, dropna=False, group_keys=False).apply(filtrar_outliers_por_grupo)

df['LOG_TARGET_CREDIT_LIMIT'] = np.log1p(df['TARGET_CREDIT_LIMIT'])
df = df.drop('TARGET_CREDIT_LIMIT', axis=1)

df.to_csv('/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_analise_exploratoria/base_para_treinamento.csv', index=False)

# Análise Preditiva

In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.optimize import minimize
import warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/bases_limpas/bases_unificadas.csv")

## Novo Tratamento e Processo de Clustering

In [7]:
y_raw = df["TARGET_CREDIT_LIMIT"].values
y_log = np.log1p(y_raw)

kf = KFold(5, shuffle=True, random_state=42)

top30_features = [
    "INCOME_PCT", "INCOME_SQRT", "AMT_INCOME_TOTAL", "INCOME_CB", "INCOME_SQ",
    "INCOME_LOG", "INCOME_PCT_x_EXT2", "AMT_REQ_CREDIT_BUREAU_YEAR", "EMPLOYED_x_EXT2",
    "INCOME_LOG_PC", "INCOME_PER_CHILD", "INCOME_x_INCTPP", "EXT_2x3", "INCOME_x_EXTMEAN",
    "INCOME_x_EMPLOYED", "FLAG_OWN_REALTY", "YEARS_EMPLOYED", "EXT_MEAN", "INCOME_LOG_PP",
    "INCOME_PER_PERSON", "EXT_SOURCE_3", "ORGANIZATION_TYPE_TE_P75", "ORGANIZATION_TYPE_TE_MED",
    "ORGANIZATION_TYPE_TE", "EMPLOYED_RATIO", "NAME_INCOME_TYPE_TE", "NAME_INCOME_TYPE_TE_MED",
    "NAME_INCOME_TYPE_TE_P75", "EXT_MIN", "EXT_SOURCE_2"
]

te_cols = ["NAME_INCOME_TYPE", "ORGANIZATION_TYPE"]
gm = y_log.mean()
te_maps, te_maps_med, te_maps_p75 = {}, {}, {}

for col in te_cols:
    m = df.groupby(col)["TARGET_CREDIT_LIMIT"].apply(lambda x: np.log1p(x).mean())
    me = df.groupby(col)["TARGET_CREDIT_LIMIT"].apply(lambda x: np.log1p(x).median())
    p7 = df.groupby(col)["TARGET_CREDIT_LIMIT"].apply(lambda x: np.log1p(x).quantile(0.75))
    c = df.groupby(col)["TARGET_CREDIT_LIMIT"].count()
    te_maps[col] = (c * m + 10 * gm) / (c + 10)
    te_maps_med[col] = (c * me + 10 * gm) / (c + 10)
    te_maps_p75[col] = (c * p7 + 10 * gm) / (c + 10)

_income_sorted = np.sort(df["AMT_INCOME_TOTAL"].dropna().values)
_ext2_sorted = np.sort(df["EXT_SOURCE_2"].fillna(0).values)
_N_combined = len(_income_sorted)

def build_X(d_in):
    d = d_in.copy()
    d["AGE_YEARS"] = (-d["DAYS_BIRTH"]) / 365
    d["YEARS_EMPLOYED"] = np.where(d["DAYS_EMPLOYED"] > 0, 0, -d["DAYS_EMPLOYED"] / 365)
    d["INCOME_LOG"] = np.log1p(d["AMT_INCOME_TOTAL"])
    d["INCOME_SQRT"] = np.sqrt(d["AMT_INCOME_TOTAL"])
    d["INCOME_PER_PERSON"] = d["AMT_INCOME_TOTAL"] / d["CNT_FAM_MEMBERS"].fillna(1).replace(0, 1)
    d["INCOME_PER_CHILD"] = d["AMT_INCOME_TOTAL"] / (d["CNT_CHILDREN"] + 1)
    d["INCOME_LOG_PP"] = np.log1p(d["INCOME_PER_PERSON"])
    d["INCOME_LOG_PC"] = np.log1p(d["INCOME_PER_CHILD"])
    
    ext = d[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    d["EXT_MEAN"] = ext.mean(axis=1)
    d["EXT_MIN"] = ext.min(axis=1)
    d["EXT_2x3"] = d["EXT_SOURCE_2"] * d["EXT_SOURCE_3"]
    
    d["INCOME_SQ"] = d["INCOME_LOG"] ** 2
    d["INCOME_CB"] = d["INCOME_LOG"] ** 3
    d["INCOME_x_EXTMEAN"] = d["INCOME_LOG"] * d["EXT_MEAN"].fillna(0)
    d["INCOME_x_EMPLOYED"] = d["INCOME_LOG"] * d["YEARS_EMPLOYED"]
    d["INCOME_x_INCTPP"] = d["INCOME_LOG"] * d["INCOME_LOG_PP"]
    d["EMPLOYED_x_EXT2"] = d["YEARS_EMPLOYED"] * d["EXT_SOURCE_2"].fillna(0)
    d["EMPLOYED_RATIO"] = d["YEARS_EMPLOYED"] / (d["AGE_YEARS"] + 1e-5)
    
    d["INCOME_PCT"] = d["AMT_INCOME_TOTAL"].rank(pct=True)
    d["EXT2_PCT"] = d["EXT_SOURCE_2"].fillna(0).rank(pct=True)
    d["INCOME_PCT_x_EXT2"] = d["INCOME_PCT"] * d["EXT2_PCT"]
    
    d["FLAG_OWN_REALTY"] = d["FLAG_OWN_REALTY"].map({"Y": 1, "N": 0}).fillna(0)
    
    for col in te_cols:
        d[col + "_TE"] = d[col].map(te_maps[col]).fillna(gm)
        d[col + "_TE_MED"] = d[col].map(te_maps_med[col]).fillna(gm)
        d[col + "_TE_P75"] = d[col].map(te_maps_p75[col]).fillna(gm)
        
    return d[top30_features]

Xdf_tr = build_X(df)

imp = SimpleImputer(strategy="median")
X_tr = imp.fit_transform(Xdf_tr)
X_tr_raw = Xdf_tr.values.astype(float)

sc = StandardScaler()
Xs_tr = sc.fit_transform(X_tr)

n = len(top30_features)
corrs = np.abs([np.corrcoef(X_tr[:, i], y_log)[0, 1] for i in range(n)])

sp = SplineTransformer(n_knots=8, degree=3, include_bias=False)
Xsp_tr = sp.fit_transform(X_tr)
sp_sc = StandardScaler()
Xsp_tr_s = sp_sc.fit_transform(Xsp_tr)
XF_tr = np.hstack([Xs_tr, Xsp_tr_s])

core_idx = np.argsort(corrs)[::-1][:15]
X_core_tr = Xs_tr[:, core_idx]
BEST_K = 8
km = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
km.fit(X_core_tr)

cluster_tr = km.predict(X_core_tr)
cluster_mean = np.array([y_log[cluster_tr == k].mean() for k in range(BEST_K)])
cluster_std = np.array([y_log[cluster_tr == k].std() for k in range(BEST_K)])
dist_tr = km.transform(X_core_tr)
oh_tr = np.eye(BEST_K)[cluster_tr]

inc_idx = top30_features.index("INCOME_LOG")
income_tr = Xs_tr[:, inc_idx:inc_idx+1]
cluster_extra_tr = np.hstack([cluster_mean[cluster_tr].reshape(-1, 1), cluster_std[cluster_tr].reshape(-1, 1), dist_tr, oh_tr, oh_tr * income_tr])

ce_sc = StandardScaler()
cluster_extra_tr_s = ce_sc.fit_transform(cluster_extra_tr)
XFC_tr = np.hstack([XF_tr, cluster_extra_tr_s])

X_tree_tr = np.hstack([X_tr_raw, cluster_extra_tr])

## Treinamento do Modelo

### Regressão Ridge

In [8]:
models_ridge = []
oof_ridge = np.zeros(len(y_raw))
for tr_idx, val_idx in kf.split(XF_tr):
    m = Ridge(alpha=100)
    m.fit(XF_tr[tr_idx], y_log[tr_idx])
    models_ridge.append(m)
    oof_ridge[val_idx] = np.expm1(m.predict(XF_tr[val_idx]))

### Regressão Ridge + Clusterização

In [9]:
models_cluster = []
oof_cluster = np.zeros(len(y_raw))
for tr_idx, val_idx in kf.split(XFC_tr):
    m = Ridge(alpha=100)
    m.fit(XFC_tr[tr_idx], y_log[tr_idx])
    models_cluster.append(m)
    oof_cluster[val_idx] = np.expm1(m.predict(XFC_tr[val_idx]))

### HistGBM

In [10]:
models_gbm = []
oof_gbm = np.zeros(len(y_raw))
for tr_idx, val_idx in kf.split(X_tree_tr):
    m = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_depth=4, min_samples_leaf=20, random_state=42)
    m.fit(X_tree_tr[tr_idx], y_log[tr_idx])
    models_gbm.append(m)
    oof_gbm[val_idx] = np.expm1(m.predict(X_tree_tr[val_idx]))

### Ensemble

In [11]:
def neg_mse_blend(w):
    w = np.abs(w) / np.abs(w).sum()
    return mean_squared_error(y_raw, w[0]*oof_ridge + w[1]*oof_cluster + w[2]*oof_gbm)

res = minimize(neg_mse_blend, [1, 1, 2], method="Nelder-Mead")
w = np.abs(res.x) / np.abs(res.x).sum()

### Finalização + Salvar arquivo .pkl

In [ ]:
artifacts = {
    "top30_features": top30_features,
    "te_cols": te_cols, "gm": gm,
    "te_maps": te_maps, "te_maps_med": te_maps_med, "te_maps_p75": te_maps_p75,
    "_income_sorted": _income_sorted, "_ext2_sorted": _ext2_sorted, "_N_combined": _N_combined,
    "inc_idx": inc_idx, "core_idx": core_idx,
    "imp": imp, "sc": sc, "sp": sp, "sp_sc": sp_sc,
    "km": km, "cluster_mean": cluster_mean, "cluster_std": cluster_std, "ce_sc": ce_sc,
    "models_ridge": models_ridge,
    "models_cluster": models_cluster,
    "models_gbm": models_gbm,
    "weights": w,
    "kf_splits": kf.n_splits
}

joblib.dump(artifacts, "modelo_credito_producao.pkl")

# Teste Cego

In [13]:
import pandas as pd
import numpy as np
import joblib
import warnings; warnings.filterwarnings("ignore")

caminho_arquivo_cego = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/teste_10k_cego.csv'
test = pd.read_csv(caminho_arquivo_cego)

## Carregamento do Modelo

In [ ]:
ARTEFATO_PATH = "/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/modelo_credito_producao.pkl"
artifacts = joblib.load(ARTEFATO_PATH)

top30_features = artifacts["top30_features"]
te_cols = artifacts["te_cols"]
gm = artifacts["gm"]
te_maps = artifacts["te_maps"]
te_maps_med = artifacts["te_maps_med"]
te_maps_p75 = artifacts["te_maps_p75"]
_income_sorted = artifacts["_income_sorted"]
_ext2_sorted = artifacts["_ext2_sorted"]
_N_combined = artifacts["_N_combined"]

imp = artifacts["imp"]
sc = artifacts["sc"]
sp = artifacts["sp"]
sp_sc = artifacts["sp_sc"]
km = artifacts["km"]
cluster_mean = artifacts["cluster_mean"]
cluster_std = artifacts["cluster_std"]
ce_sc = artifacts["ce_sc"]
core_idx = artifacts["core_idx"]
inc_idx = artifacts["inc_idx"]
models_ridge = artifacts["models_ridge"]
models_cluster = artifacts["models_cluster"]
models_gbm = artifacts["models_gbm"]
w = artifacts["weights"]
n_splits = artifacts["kf_splits"]
BEST_K = len(cluster_mean)

## Tratamento do Teste Cego

In [15]:
def build_X_test(d_in):
    d = d_in.copy()
    d["AGE_YEARS"] = (-d["DAYS_BIRTH"]) / 365
    d["YEARS_EMPLOYED"] = np.where(d["DAYS_EMPLOYED"] > 0, 0, -d["DAYS_EMPLOYED"] / 365)
    d["INCOME_LOG"] = np.log1p(d["AMT_INCOME_TOTAL"])
    d["INCOME_SQRT"] = np.sqrt(d["AMT_INCOME_TOTAL"])
    d["INCOME_PER_PERSON"] = d["AMT_INCOME_TOTAL"] / d["CNT_FAM_MEMBERS"].fillna(1).replace(0, 1)
    d["INCOME_PER_CHILD"] = d["AMT_INCOME_TOTAL"] / (d["CNT_CHILDREN"] + 1)
    d["INCOME_LOG_PP"] = np.log1p(d["INCOME_PER_PERSON"])
    d["INCOME_LOG_PC"] = np.log1p(d["INCOME_PER_CHILD"])
    
    ext = d[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    d["EXT_MEAN"] = ext.mean(axis=1)
    d["EXT_MIN"] = ext.min(axis=1)
    d["EXT_2x3"] = d["EXT_SOURCE_2"] * d["EXT_SOURCE_3"]
    
    d["INCOME_SQ"] = d["INCOME_LOG"] ** 2
    d["INCOME_CB"] = d["INCOME_LOG"] ** 3
    d["INCOME_x_EXTMEAN"] = d["INCOME_LOG"] * d["EXT_MEAN"].fillna(0)
    d["INCOME_x_EMPLOYED"] = d["INCOME_LOG"] * d["YEARS_EMPLOYED"]
    d["INCOME_x_INCTPP"] = d["INCOME_LOG"] * d["INCOME_LOG_PP"]
    d["EMPLOYED_x_EXT2"] = d["YEARS_EMPLOYED"] * d["EXT_SOURCE_2"].fillna(0)
    d["EMPLOYED_RATIO"] = d["YEARS_EMPLOYED"] / (d["AGE_YEARS"] + 1e-5)
    
    d["INCOME_PCT"] = np.searchsorted(_income_sorted, d_in["AMT_INCOME_TOTAL"].fillna(_income_sorted[0]).values, side="right") / _N_combined
    d["EXT2_PCT"] = np.searchsorted(_ext2_sorted, d_in["EXT_SOURCE_2"].fillna(0).values, side="right") / _N_combined

    d["INCOME_PCT_x_EXT2"] = d["INCOME_PCT"] * d["EXT2_PCT"]
    d["FLAG_OWN_REALTY"] = d["FLAG_OWN_REALTY"].map({"Y": 1, "N": 0}).fillna(0)
    
    for col in te_cols:
        d[col + "_TE"] = d[col].map(te_maps[col]).fillna(gm)
        d[col + "_TE_MED"] = d[col].map(te_maps_med[col]).fillna(gm)
        d[col + "_TE_P75"] = d[col].map(te_maps_p75[col]).fillna(gm)
        
    return d[top30_features]

## Predição do Teste Pelo Modelo

In [ ]:
Xdf_te = build_X_test(test)

X_te = imp.transform(Xdf_te)
X_te_raw = Xdf_te.values.astype(float)
Xs_te = sc.transform(X_te)

Xsp_te = sp.transform(X_te)
Xsp_te_s = sp_sc.transform(Xsp_te)
XF_te = np.hstack([Xs_te, Xsp_te_s])

X_core_te = Xs_te[:, core_idx]
cluster_te = km.predict(X_core_te)
dist_te = km.transform(X_core_te)
oh_te = np.eye(BEST_K)[cluster_te]
income_te = Xs_te[:, inc_idx:inc_idx+1]

cluster_extra_te = np.hstack([
    cluster_mean[cluster_te].reshape(-1, 1), 
    cluster_std[cluster_te].reshape(-1, 1), 
    dist_te, oh_te, oh_te * income_te
])

cluster_extra_te_s = ce_sc.transform(cluster_extra_te)
XFC_te = np.hstack([XF_te, cluster_extra_te_s])
X_tree_te = np.hstack([X_te_raw, cluster_extra_te])

print("Realizando predições...")
preds_ridge = np.zeros(len(X_te))
for model in models_ridge:
    preds_ridge += np.expm1(model.predict(XF_te)) / n_splits

preds_cluster = np.zeros(len(X_te))
for model in models_cluster:
    preds_cluster += np.expm1(model.predict(XFC_te)) / n_splits

preds_gbm = np.zeros(len(X_te))
for model in models_gbm:
    preds_gbm += np.expm1(model.predict(X_tree_te)) / n_splits

pred_ens = w[0]*preds_ridge + w[1]*preds_cluster + w[2]*preds_gbm

out = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"],
    "TARGET_CREDIT_LIMIT": np.round(pred_ens, 2)
})

saida_nome = "predicoes_finais.csv"
out.to_csv(saida_nome, index=False)
print(f"✅ Predições salvas com sucesso em: '{saida_nome}'.")